[![pythonista](img/pythonista.png)](https://www.pythonista.io)

# Soluciones de los ejercicios.

## Configuración.

Ejecuta la siguiente celda antes que cualquier otra. Abre en modo de escritura la base de trabajo ```data/curso.duckdb``` y adjunta los datos del curso en sólo lectura.

In [ ]:
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.displaycon = False

conexion = duckdb.connect('data/curso.duckdb')
conexion.execute("ATTACH IF NOT EXISTS 'data/thelook.duckdb' AS thelook (READ_ONLY)")
for tabla in ['distribution_centers', 'products', 'users', 'orders', 'order_items']:
    conexion.execute(f"CREATE OR REPLACE VIEW {tabla} AS SELECT * FROM thelook.main.{tabla}")
%sql conexion

Este notebook usa la configuración de escritura de las lecciones 7 y 8. **Ejecuta antes esas dos lecciones**, porque algunas soluciones parten de las tablas que se crean en ellas, y cierra sus *kernels* para liberar el archivo ```data/curso.duckdb```.

En la mayoría de los ejercicios hay más de una solución correcta.

## Lección 1. Introducción.

**1.** Columnas de ```orders``` y su tipo. Son ```TIMESTAMP```: ```created_at```, ```returned_at```, ```shipped_at``` y ```delivered_at```.

In [ ]:
%%sql
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_catalog = 'thelook' AND table_name = 'orders'
ORDER BY ordinal_position;

**2.** Número de tablas.

In [ ]:
%%sql
SELECT COUNT(*) AS tablas
FROM information_schema.tables
WHERE table_catalog = 'thelook' AND table_type = 'BASE TABLE';

**3.**

In [ ]:
%%sql
SELECT id, name, brand
FROM products
ORDER BY id
FETCH FIRST 20 ROWS ONLY;

## Lección 2. Consultas básicas.

**1.**

In [ ]:
%%sql
SELECT name, brand, retail_price
FROM products
WHERE department = 'Women' AND retail_price < 10
ORDER BY retail_price;

**2.**

In [ ]:
%%sql
SELECT DISTINCT country
FROM users
ORDER BY country;

**3.**

In [ ]:
%%sql
SELECT order_id, user_id, status
FROM orders
WHERE status IN ('Cancelled', 'Returned')
ORDER BY order_id
FETCH FIRST 20 ROWS ONLY;

**4.** La condición sobre el margen se escribe con la expresión completa, porque el alias no puede usarse en ```WHERE```.

In [ ]:
%%sql
SELECT name, retail_price, cost, ROUND(retail_price - cost, 2) AS margen
FROM products
WHERE LOWER(name) LIKE '%hoodie%'
  AND retail_price - cost > 50
ORDER BY margen DESC;

**5.**

In [ ]:
%%sql
SELECT first_name, last_name, age, state
FROM users
WHERE country = 'Brasil'
ORDER BY age DESC, last_name
FETCH FIRST 5 ROWS ONLY;

## Lección 3. Agregaciones.

**1.**

In [ ]:
%%sql
SELECT country, COUNT(*) AS clientes
FROM users
GROUP BY country
ORDER BY clientes DESC;

**2.**

In [ ]:
%%sql
SELECT traffic_source, COUNT(*) AS clientes, ROUND(AVG(age), 1) AS edad_promedio
FROM users
GROUP BY traffic_source
ORDER BY clientes DESC;

**3.**

In [ ]:
%%sql
SELECT category, COUNT(*) AS productos, ROUND(AVG(retail_price), 2) AS precio_promedio
FROM products
GROUP BY category
HAVING AVG(retail_price) > 60
ORDER BY precio_promedio DESC;

**4.** y **5.** Conteo condicional y tasa de devolución. Se multiplica por ```100.0``` para que la división no sea entera.

In [ ]:
%%sql
SELECT gender,
  COUNT(*) AS ordenes,
  COUNT(CASE WHEN status = 'Returned' THEN 1 END) AS devueltas,
  ROUND(100.0 * COUNT(CASE WHEN status = 'Returned' THEN 1 END) / COUNT(*), 1) AS tasa_devolucion_pct
FROM orders
GROUP BY gender;

## Lección 4. Combinación de tablas y lógica condicional.

**1.**

In [ ]:
%%sql
SELECT p.brand, ROUND(SUM(oi.sale_price), 2) AS ventas
FROM order_items AS oi
JOIN products AS p ON oi.product_id = p.id
WHERE oi.status = 'Complete'
GROUP BY p.brand
ORDER BY ventas DESC
FETCH FIRST 10 ROWS ONLY;

**2.** Patrón *anti-join*: ```LEFT JOIN``` y conservar las filas sin coincidencia.

In [ ]:
%%sql
SELECT p.name, p.category
FROM products AS p
LEFT JOIN order_items AS oi ON p.id = oi.product_id
WHERE oi.id IS NULL
ORDER BY p.category, p.name;

**3.** Como se suma ```sale_price```, se parte de ```order_items```; las órdenes se cuentan con ```COUNT(DISTINCT ...)``` porque una orden tiene varios artículos.

In [ ]:
%%sql
SELECT u.traffic_source,
  COUNT(DISTINCT oi.order_id) AS ordenes,
  ROUND(SUM(oi.sale_price), 2) AS total_vendido
FROM order_items AS oi
JOIN users AS u ON oi.user_id = u.id
GROUP BY u.traffic_source
ORDER BY total_vendido DESC;

**4.**

In [ ]:
%%sql
SELECT
  CASE
    WHEN num_of_item = 1 THEN '1 artículo'
    WHEN num_of_item <= 3 THEN '2-3 artículos'
    ELSE '4 o más'
  END AS grupo,
  COUNT(*) AS ordenes
FROM orders
GROUP BY
  CASE
    WHEN num_of_item = 1 THEN '1 artículo'
    WHEN num_of_item <= 3 THEN '2-3 artículos'
    ELSE '4 o más'
  END
ORDER BY grupo;

**5.**

In [ ]:
%%sql
SELECT dc.name AS centro_distribucion,
  COUNT(CASE WHEN oi.status = 'Complete' THEN 1 END) AS completados,
  COUNT(CASE WHEN oi.status = 'Returned' THEN 1 END) AS devueltos
FROM order_items AS oi
JOIN products AS p ON oi.product_id = p.id
JOIN distribution_centers AS dc ON p.distribution_center_id = dc.id
GROUP BY dc.name
ORDER BY completados DESC;

## Lección 5. Fechas, subconsultas y expresiones de tabla comunes.

**1.**

In [ ]:
%%sql
SELECT EXTRACT(YEAR FROM created_at) AS anio, COUNT(*) AS clientes
FROM users
GROUP BY EXTRACT(YEAR FROM created_at)
ORDER BY anio;

**2.** El mejor mes queda en la primera fila.

In [ ]:
%%sql
SELECT EXTRACT(MONTH FROM created_at) AS mes, ROUND(SUM(sale_price), 2) AS ventas
FROM order_items
WHERE status = 'Complete'
  AND created_at >= TIMESTAMP '2024-01-01 00:00:00'
  AND created_at < TIMESTAMP '2025-01-01 00:00:00'
GROUP BY EXTRACT(MONTH FROM created_at)
ORDER BY ventas DESC;

**3.**

In [ ]:
%%sql
SELECT EXTRACT(YEAR FROM created_at) AS anio,
  ROUND(AVG(EXTRACT(DAY FROM (shipped_at - created_at))
            + EXTRACT(HOUR FROM (shipped_at - created_at)) / 24.0), 2) AS dias_promedio_envio
FROM orders
WHERE shipped_at IS NOT NULL
GROUP BY EXTRACT(YEAR FROM created_at)
ORDER BY anio;

**4.**

In [ ]:
%%sql
SELECT category, ROUND(AVG(retail_price), 2) AS precio_promedio
FROM products
GROUP BY category
HAVING AVG(retail_price) > (SELECT AVG(retail_price) FROM products)
ORDER BY precio_promedio DESC;

**5.**

In [ ]:
%%sql
WITH mensual AS (
  SELECT EXTRACT(MONTH FROM created_at) AS mes, SUM(sale_price) AS ventas
  FROM order_items
  WHERE status = 'Complete'
    AND created_at >= TIMESTAMP '2024-01-01 00:00:00'
    AND created_at < TIMESTAMP '2025-01-01 00:00:00'
  GROUP BY EXTRACT(MONTH FROM created_at)
)
SELECT mes, ROUND(ventas, 2) AS ventas
FROM mensual
WHERE ventas > (SELECT AVG(ventas) FROM mensual)
ORDER BY mes;

## Lección 6. Funciones de ventana.

**1.** Con ```RANK()```, las marcas empatadas en el quinto lugar aparecen todas.

In [ ]:
%%sql
WITH conteo AS (
  SELECT department, brand, COUNT(*) AS productos
  FROM products
  WHERE brand IS NOT NULL
  GROUP BY department, brand
),
ranking AS (
  SELECT department, brand, productos,
    RANK() OVER (PARTITION BY department ORDER BY productos DESC) AS posicion
  FROM conteo
)
SELECT department, posicion, brand, productos
FROM ranking
WHERE posicion <= 5
ORDER BY department, posicion, brand;

**2.**

In [ ]:
%%sql
WITH numeradas AS (
  SELECT user_id, order_id, created_at,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at, order_id) AS rn
  FROM orders
)
SELECT user_id, order_id, created_at AS primera_orden
FROM numeradas
WHERE rn = 1
ORDER BY user_id
FETCH FIRST 20 ROWS ONLY;

**3.**

In [ ]:
%%sql
SELECT brand, name, retail_price,
  DENSE_RANK() OVER (PARTITION BY brand ORDER BY retail_price DESC) AS posicion_en_marca
FROM products
ORDER BY brand, posicion_en_marca, name
FETCH FIRST 20 ROWS ONLY;

**4.**

In [ ]:
%%sql
WITH nuevos AS (
  SELECT EXTRACT(YEAR FROM created_at) AS anio,
    EXTRACT(MONTH FROM created_at) AS mes,
    COUNT(*) AS clientes_nuevos
  FROM users
  GROUP BY EXTRACT(YEAR FROM created_at), EXTRACT(MONTH FROM created_at)
)
SELECT anio, mes, clientes_nuevos,
  SUM(clientes_nuevos) OVER (ORDER BY anio, mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS clientes_acumulados
FROM nuevos
ORDER BY anio, mes;

**5.**

In [ ]:
%%sql
WITH mensual AS (
  SELECT p.category AS categoria,
    EXTRACT(YEAR FROM oi.created_at) AS anio,
    EXTRACT(MONTH FROM oi.created_at) AS mes,
    SUM(oi.sale_price) AS ventas
  FROM order_items AS oi
  JOIN products AS p ON oi.product_id = p.id
  WHERE oi.status = 'Complete'
  GROUP BY p.category, EXTRACT(YEAR FROM oi.created_at), EXTRACT(MONTH FROM oi.created_at)
),
con_anterior AS (
  SELECT categoria, anio, mes, ventas,
    LAG(ventas) OVER (PARTITION BY categoria ORDER BY anio, mes) AS ventas_anterior
  FROM mensual
)
SELECT categoria, anio, mes,
  ROUND(ventas, 2) AS ventas,
  ROUND((ventas - ventas_anterior) / NULLIF(ventas_anterior, 0) * 100, 1) AS crecimiento_pct
FROM con_anterior
ORDER BY categoria, anio, mes
FETCH FIRST 20 ROWS ONLY;

## Lección 7. Tablas, vistas y carga de datos.

Cada solución elimina primero el objeto, si existe, para poder ejecutarse varias veces.

**1.**

In [ ]:
%%sql
DROP TABLE IF EXISTS analitica.resumen_clientes;
CREATE TABLE analitica.resumen_clientes AS
SELECT user_id,
  COUNT(*) AS ordenes,
  MIN(created_at) AS primera_orden,
  MAX(created_at) AS ultima_orden
FROM orders
GROUP BY user_id;
SELECT user_id, ordenes, primera_orden, ultima_orden
FROM analitica.resumen_clientes
ORDER BY ordenes DESC, user_id
FETCH FIRST 20 ROWS ONLY;

**2.**

In [ ]:
%%sql
DROP VIEW IF EXISTS analitica.v_productos_sin_venta;
CREATE VIEW analitica.v_productos_sin_venta AS
SELECT p.id, p.name, p.category
FROM products AS p
WHERE NOT EXISTS (
  SELECT 1 FROM order_items AS oi WHERE oi.product_id = p.id
);
SELECT COUNT(*) AS productos_sin_venta FROM analitica.v_productos_sin_venta;

**3.** ```NOT EXISTS``` evita insertar metas que ya existen (y que violarían la clave primaria). Por lo mismo, la sentencia puede ejecutarse varias veces sin error.

In [ ]:
%%sql
INSERT INTO analitica.metas_mensuales (anio, mes, categoria, meta_ventas)
SELECT 2026, 1, v.categoria, ROUND(v.ventas * 1.15, 2)
FROM analitica.ventas_mensuales AS v
WHERE v.anio = 2025 AND v.mes = 1
  AND NOT EXISTS (
    SELECT 1
    FROM analitica.metas_mensuales AS m
    WHERE m.anio = 2026 AND m.mes = 1 AND m.categoria = v.categoria
  );
SELECT * FROM analitica.metas_mensuales WHERE anio = 2026 ORDER BY mes, categoria;

**4.**

In [ ]:
%%sql
MERGE INTO analitica.ventas_mensuales AS destino
USING (
  SELECT EXTRACT(YEAR FROM oi.created_at) AS anio,
    EXTRACT(MONTH FROM oi.created_at) AS mes,
    p.category AS categoria,
    COUNT(*) AS articulos,
    SUM(oi.sale_price) AS ventas
  FROM order_items AS oi
  JOIN products AS p ON oi.product_id = p.id
  WHERE oi.status IN ('Complete', 'Shipped')
    AND oi.created_at >= TIMESTAMP '2025-12-01 00:00:00'
    AND oi.created_at < TIMESTAMP '2026-01-01 00:00:00'
  GROUP BY EXTRACT(YEAR FROM oi.created_at), EXTRACT(MONTH FROM oi.created_at), p.category
) AS lote
  ON destino.anio = lote.anio AND destino.mes = lote.mes AND destino.categoria = lote.categoria
WHEN MATCHED THEN
  UPDATE SET articulos = lote.articulos, ventas = lote.ventas
WHEN NOT MATCHED THEN
  INSERT (anio, mes, categoria, articulos, ventas)
  VALUES (lote.anio, lote.mes, lote.categoria, lote.articulos, lote.ventas);
SELECT anio, mes, categoria, articulos, ROUND(ventas, 2) AS ventas
FROM analitica.ventas_mensuales
WHERE anio = 2025 AND mes = 12
ORDER BY ventas DESC;

## Lección 8. Denormalización.

**1.**

In [ ]:
%%sql
SELECT marca, ROUND(SUM(precio_venta - costo), 2) AS margen_bruto
FROM modelo.ventas_detalle
WHERE estado_articulo = 'Complete'
GROUP BY marca
ORDER BY margen_bruto DESC
FETCH FIRST 10 ROWS ONLY;

**2.** El ticket es la venta total entre el número de **órdenes** distintas, no entre el número de filas (artículos).

In [ ]:
%%sql
SELECT cliente_pais,
  COUNT(DISTINCT order_id) AS ordenes,
  ROUND(SUM(precio_venta) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS ticket_promedio
FROM modelo.ventas_detalle
GROUP BY cliente_pais
ORDER BY ticket_promedio DESC;

**3.** Una tabla ancha con granularidad de orden tendría:

* **Atributos de la orden:** ```order_id```, ```status```, ```created_at```, ```shipped_at```, ```delivered_at```.
* **Atributos del cliente:** ```user_id```, género, edad, país, estado, origen de tráfico.
* **Medidas agregadas de los artículos:** número de artículos, venta total, costo total, margen, número de categorías distintas.

Los atributos de producto y de centro de distribución **no caben** en esta granularidad, porque una orden puede tener varios productos: se resumen como medidas o se omiten.

**4.**

In [ ]:
%%sql
DROP TABLE IF EXISTS modelo.ordenes_detalle;
CREATE TABLE modelo.ordenes_detalle AS
SELECT o.order_id,
  o.status,
  o.created_at,
  o.delivered_at,
  u.id AS cliente_id,
  u.gender AS cliente_genero,
  u.age AS cliente_edad,
  u.country AS cliente_pais,
  u.traffic_source AS cliente_origen,
  COUNT(oi.id) AS articulos,
  COUNT(DISTINCT p.category) AS categorias,
  SUM(oi.sale_price) AS venta_total,
  SUM(p.cost) AS costo_total,
  SUM(oi.sale_price - p.cost) AS margen
FROM orders AS o
JOIN users AS u ON o.user_id = u.id
LEFT JOIN order_items AS oi ON o.order_id = oi.order_id
LEFT JOIN products AS p ON oi.product_id = p.id
GROUP BY o.order_id, o.status, o.created_at, o.delivered_at,
  u.id, u.gender, u.age, u.country, u.traffic_source;
SELECT
  (SELECT COUNT(*) FROM orders) AS filas_orders,
  (SELECT COUNT(*) FROM modelo.ordenes_detalle) AS filas_tabla_ancha;

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2026.</p>